# Análisis de Armadura en Forma de Arco con OpenSeesPy

Este notebook modela y analiza una estructura de armadura en forma de arco 3D basada en una descripción paramétrica. Cubriremos:
1. Definición de Parámetros
2. Generación de Geometría (Nodos y Elementos)
3. Definición de Apoyos y Cargas
4. Ejecución de Análisis Estático
5. Visualización de Resultados 3D

## 1. PREPARACIÓN Y CONFIGURACIÓN INICIAL

In [ ]:
# Instalación de Librerías (ejecutar una vez si no están instaladas)
!pip install numpy pandas matplotlib openseespy opsvis

In [ ]:
# Importación de librerías
import openseespy.opensees as ops
import opsvis as opsv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Limpieza y Definición del Modelo
ops.wipe()
ops.model('basic', '-ndm', 3, '-ndf', 6)

## 2. Definición de Parámetros de la Armadura de Arco

In [ ]:
# Parámetros Generales
L = 17.50  # Claro (m)
f = 1.50   # Flecha (m)
S = 5.50   # Separación entre armaduras (m)
N = 6      # Número de armaduras

# Ecuación del arco: y = a * x^2, pero el origen se tomará en el apoyo izquierdo.
# La ecuación con origen en (0,0) y vértice en (L/2, f) es: y = (4*f/L**2) * x * (L-x)
def parabolic_arch_y(x, L, f):
    return (4 * f / L**2) * x * (L - x)

# Material: Acero A36
fy = 250e6    # Límite de fluencia (Pa = N/m^2)
E = 200e9     # Módulo de Elasticidad (Pa = N/m^2)
G = E / (2 * (1 + 0.3)) # Módulo de corte (asumiendo Poisson = 0.3)

# ---- Propiedades de los Perfiles ----
# Unidades en metros para consistencia con OpenSees (m, N, s)

# Cordones (2 ángulos de 2"x2"x1/4")
# Área total de 2 ángulos
A_cordon = 0.002258 # m^2
# Inercia y torsión para el elemento compuesto (aproximación)
# Para un análisis de armadura (truss), solo se necesita el área.
# Para un análisis de pórtico (beam-column), se necesitan las inercias.
# Asumiremos que los dos ángulos actúan juntos.
Iy_cordon = 2 * (6.77e-8) # m^4 (débil, suma simple)
Iz_cordon = 2 * (6.77e-8 + 11.29e-4 * (0.025)**2) # m^4 (fuerte, usando teorema de ejes paralelos, suponiendo una separación de 5cm)
J_cordon = 2 * (6.77e-8) # m^4 (aproximación simple)

# Cajón y Diagonales (1 ángulo de 1"x1"x1/4")
A_diag = 5.81e-4 # m^2
Iy_diag = 1.13e-8 # m^4
Iz_diag = 1.13e-8 # m^4
J_diag = 1.13e-8 # m^4 (aproximación)

# Dimensiones de la sección de la armadura
truss_width = 0.20 # Ancho del cajón (m, en dirección Y global)
truss_height = 0.45 # Alto del cajón (m, en dirección Z local de la armadura)

print("Parámetros de la estructura definidos.")
print(f"Material: Acero A36, E = {E/1e9} GPa")
print(f"Área de Cordón (2 ángulos): {A_cordon} m^2")
print(f"Área de Diagonal/Cajón (1 ángulo): {A_diag} m^2")

## 3. Generación de Nodos de una Sola Armadura (en el plano XY)

Primero, generaremos todos los nodos para una sola armadura en el plano XZ (asumiendo que Z es la dirección vertical de la armadura) y luego rotaremos/moveremos el conjunto. Para simplificar, construiremos la armadura en el plano XY (X a lo largo del claro, Y vertical) y luego la moveremos/rotaremos a su posición 3D final.

In [ ]:
# Parámetros para la generación de nodos
num_sections = 22 # Número de secciones del cajón
dx = L / num_sections # Distancia horizontal entre secciones

node_tag_counter = 1
truss_nodes = {}

print(f"Generando nodos para una armadura con {num_sections+1} estaciones de nodos...")

# Generar nodos para una armadura en el plano XY (Z=0)
for i in range(num_sections + 1):
    x = i * dx
    # Altura del cordón superior según la parábola
    y_upper = parabolic_arch_y(x, L, f)
    # Altura del cordón inferior
    y_lower = y_upper - truss_height
    
    # Nodos para esta estación (i)
    # Nodos del cordón superior (delantero y trasero)
    # Trasero (Y = -truss_width/2), Delantero (Y = +truss_width/2)
    # Lo construiremos en el plano XZ, por lo que Y será la profundidad.
    # x -> X, y -> Z, z -> Y
    
    # Cordón superior trasero (CST)
    truss_nodes[i, 'CST'] = (node_tag_counter, x, -truss_width/2, y_upper)
    ops.node(node_tag_counter, x, -truss_width/2, y_upper)
    node_tag_counter += 1
    
    # Cordón superior delantero (CSD)
    truss_nodes[i, 'CSD'] = (node_tag_counter, x, +truss_width/2, y_upper)
    ops.node(node_tag_counter, x, +truss_width/2, y_upper)
    node_tag_counter += 1

    # Cordón inferior trasero (CIT)
    truss_nodes[i, 'CIT'] = (node_tag_counter, x, -truss_width/2, y_lower)
    ops.node(node_tag_counter, x, -truss_width/2, y_lower)
    node_tag_counter += 1

    # Cordón inferior delantero (CID)
    truss_nodes[i, 'CID'] = (node_tag_counter, x, +truss_width/2, y_lower)
    ops.node(node_tag_counter, x, +truss_width/2, y_lower)
    node_tag_counter += 1

print(f"Se generaron {node_tag_counter - 1} nodos para la primera armadura.")
print("Ejemplo de nodo guardado:", truss_nodes[0, 'CST'])

# Visualizar los nodos generados para la primera armadura
plt.figure(figsize=(15, 5))
opsv.plot_model(node_labels=0, title="Nodos de una sola armadura (vista 3D)")
plt.show()

## 4. Creación de Elementos de una Sola Armadura

Ahora conectaremos los nodos con elementos `elasticBeamColumn` para formar los cordones, los cajones y las diagonales.

In [ ]:
# Transformación Geométrica
# Necesitamos una transformación para los cordones y otra para los elementos verticales/diagonales.
# El vector 'vecxz' define el plano local x-z del elemento.

# Para los cordones (elementos mayormente horizontales), el plano x-z local puede ser vertical.
ops.geomTransf('Linear', 1, 0, 0, 1) # Tag 1 para cordones

# Para los elementos verticales (del cajón), el plano x-z puede alinearse con el plano XZ global.
ops.geomTransf('Linear', 2, 1, 0, 0) # Tag 2 para verticales/diagonales

print("Transformaciones geométricas definidas.")

In [ ]:
ele_tag_counter = 1

print("Creando elementos...")

# Crear Cordones
for i in range(num_sections):
    # Cordón Superior Trasero (CST)
    n1 = truss_nodes[i, 'CST'][0]
    n2 = truss_nodes[i+1, 'CST'][0]
    ops.element('elasticBeamColumn', ele_tag_counter, n1, n2, A_cordon, E, G, J_cordon, Iy_cordon, Iz_cordon, 1)
    ele_tag_counter += 1

    # Cordón Superior Delantero (CSD)
    n1 = truss_nodes[i, 'CSD'][0]
    n2 = truss_nodes[i+1, 'CSD'][0]
    ops.element('elasticBeamColumn', ele_tag_counter, n1, n2, A_cordon, E, G, J_cordon, Iy_cordon, Iz_cordon, 1)
    ele_tag_counter += 1

    # Cordón Inferior Trasero (CIT)
    n1 = truss_nodes[i, 'CIT'][0]
    n2 = truss_nodes[i+1, 'CIT'][0]
    ops.element('elasticBeamColumn', ele_tag_counter, n1, n2, A_cordon, E, G, J_cordon, Iy_cordon, Iz_cordon, 1)
    ele_tag_counter += 1

    # Cordón Inferior Delantero (CID)
    n1 = truss_nodes[i, 'CID'][0]
    n2 = truss_nodes[i+1, 'CID'][0]
    ops.element('elasticBeamColumn', ele_tag_counter, n1, n2, A_cordon, E, G, J_cordon, Iy_cordon, Iz_cordon, 1)
    ele_tag_counter += 1

print(f"Cordones creados. Última etiqueta de elemento: {ele_tag_counter-1}")

# Crear Cajones y Diagonales
for i in range(num_sections + 1):
    # Nodos de la sección actual
    n_cst = truss_nodes[i, 'CST'][0]
    n_csd = truss_nodes[i, 'CSD'][0]
    n_cit = truss_nodes[i, 'CIT'][0]
    n_cid = truss_nodes[i, 'CID'][0]
    
    # Elementos Verticales del Cajón
    ops.element('elasticBeamColumn', ele_tag_counter, n_cst, n_cit, A_diag, E, G, J_diag, Iy_diag, Iz_diag, 2); ele_tag_counter += 1
    ops.element('elasticBeamColumn', ele_tag_counter, n_csd, n_cid, A_diag, E, G, J_diag, Iy_diag, Iz_diag, 2); ele_tag_counter += 1
    
    # Elementos Horizontales del Cajón
    ops.element('elasticBeamColumn', ele_tag_counter, n_cst, n_csd, A_diag, E, G, J_diag, Iy_diag, Iz_diag, 2); ele_tag_counter += 1
    ops.element('elasticBeamColumn', ele_tag_counter, n_cit, n_cid, A_diag, E, G, J_diag, Iy_diag, Iz_diag, 2); ele_tag_counter += 1

    # Diagonales (si no es la última sección)
    if i < num_sections:
        # Nodos de la siguiente sección
        n_cst_next = truss_nodes[i+1, 'CST'][0]
        n_csd_next = truss_nodes[i+1, 'CSD'][0]
        n_cit_next = truss_nodes[i+1, 'CIT'][0]
        n_cid_next = truss_nodes[i+1, 'CID'][0]
        
        # Diagonales en el plano trasero (CST a CIT_next)
        ops.element('elasticBeamColumn', ele_tag_counter, n_cst, n_cit_next, A_diag, E, G, J_diag, Iy_diag, Iz_diag, 2); ele_tag_counter += 1
        ops.element('elasticBeamColumn', ele_tag_counter, n_cit, n_cst_next, A_diag, E, G, J_diag, Iy_diag, Iz_diag, 2); ele_tag_counter += 1
        
        # Diagonales en el plano delantero (CSD a CID_next)
        ops.element('elasticBeamColumn', ele_tag_counter, n_csd, n_cid_next, A_diag, E, G, J_diag, Iy_diag, Iz_diag, 2); ele_tag_counter += 1
        ops.element('elasticBeamColumn', ele_tag_counter, n_cid, n_csd_next, A_diag, E, G, J_diag, Iy_diag, Iz_diag, 2); ele_tag_counter += 1

print(f"Cajones y diagonales creados. Total elementos: {ele_tag_counter-1}")

# Visualizar el modelo con elementos
plt.figure(figsize=(15, 5))
opsv.plot_model(show_nodes=False, show_ele_labels=False, title="Modelo de una sola armadura")
plt.show()

## 5. Replicación de Armaduras y Conexiones

El paso anterior creó una armadura completa, pero todos sus nodos y elementos están en el modelo. OpenSees no tiene un comando `copy` o `replicate` nativo. La forma correcta de modelar las 6 armaduras es generar los nodos y elementos para todas ellas en un gran bucle. Reiniciaremos el modelo y lo construiremos todo de una vez.

In [ ]:
# --- REINICIO Y CONSTRUCCIÓN COMPLETA ---
print("Reiniciando el modelo para construir la estructura completa de 6 armaduras...")
ops.wipe()
ops.model('basic', '-ndm', 3, '-ndf', 6)

node_tag_counter = 1
ele_tag_counter = 1
all_truss_nodes = {}

# Definir transformaciones otra vez
ops.geomTransf('Linear', 1, 0, 0, 1) # Para cordones
ops.geomTransf('Linear', 2, 1, 0, 0) # Para verticales/diagonales

print(f"Generando {N} armaduras separadas por {S} m...")

for frame_idx in range(N):
    z_offset = frame_idx * S
    
    # --- 1. Generar Nodos para esta armadura ---
    truss_nodes_frame = {}
    for i in range(num_sections + 1):
        x = i * dx
        y_arch = parabolic_arch_y(x, L, f)
        
        # Nodos del cordón superior (delantero y trasero)
        ops.node(node_tag_counter, x, -truss_width/2, y_arch + z_offset); 
        truss_nodes_frame[i, 'CST'] = (node_tag_counter, x, -truss_width/2, y_arch + z_offset); node_tag_counter += 1
        
        ops.node(node_tag_counter, x, +truss_width/2, y_arch + z_offset); 
        truss_nodes_frame[i, 'CSD'] = (node_tag_counter, x, +truss_width/2, y_arch + z_offset); node_tag_counter += 1

        # Nodos del cordón inferior
        ops.node(node_tag_counter, x, -truss_width/2, y_arch - truss_height + z_offset); 
        truss_nodes_frame[i, 'CIT'] = (node_tag_counter, x, -truss_width/2, y_arch - truss_height + z_offset); node_tag_counter += 1
        
        ops.node(node_tag_counter, x, +truss_width/2, y_arch - truss_height + z_offset); 
        truss_nodes_frame[i, 'CID'] = (node_tag_counter, x, +truss_width/2, y_arch - truss_height + z_offset); node_tag_counter += 1
        
    all_truss_nodes[frame_idx] = truss_nodes_frame
    
    # --- 2. Crear Elementos para esta armadura ---
    # Cordones
    for i in range(num_sections):
        for key in ['CST', 'CSD', 'CIT', 'CID']:
            n1 = truss_nodes_frame[i, key][0]
            n2 = truss_nodes_frame[i+1, key][0]
            ops.element('elasticBeamColumn', ele_tag_counter, n1, n2, A_cordon, E, G, J_cordon, Iy_cordon, Iz_cordon, 1); ele_tag_counter += 1

    # Cajones y Diagonales
    for i in range(num_sections + 1):
        n_cst = truss_nodes_frame[i, 'CST'][0]
        n_csd = truss_nodes_frame[i, 'CSD'][0]
        n_cit = truss_nodes_frame[i, 'CIT'][0]
        n_cid = truss_nodes_frame[i, 'CID'][0]
        
        ops.element('elasticBeamColumn', ele_tag_counter, n_cst, n_cit, A_diag, E, G, J_diag, Iy_diag, Iz_diag, 2); ele_tag_counter += 1
        ops.element('elasticBeamColumn', ele_tag_counter, n_csd, n_cid, A_diag, E, G, J_diag, Iy_diag, Iz_diag, 2); ele_tag_counter += 1
        ops.element('elasticBeamColumn', ele_tag_counter, n_cst, n_csd, A_diag, E, G, J_diag, Iy_diag, Iz_diag, 2); ele_tag_counter += 1
        ops.element('elasticBeamColumn', ele_tag_counter, n_cit, n_cid, A_diag, E, G, J_diag, Iy_diag, Iz_diag, 2); ele_tag_counter += 1
        
        if i < num_sections:
            n_cst_next = truss_nodes_frame[i+1, 'CST'][0]
            n_csd_next = truss_nodes_frame[i+1, 'CSD'][0]
            n_cit_next = truss_nodes_frame[i+1, 'CIT'][0]
            n_cid_next = truss_nodes_frame[i+1, 'CID'][0]
            
            ops.element('elasticBeamColumn', ele_tag_counter, n_cst, n_cit_next, A_diag, E, G, J_diag, Iy_diag, Iz_diag, 2); ele_tag_counter += 1
            ops.element('elasticBeamColumn', ele_tag_counter, n_cit, n_cst_next, A_diag, E, G, J_diag, Iy_diag, Iz_diag, 2); ele_tag_counter += 1
            ops.element('elasticBeamColumn', ele_tag_counter, n_csd, n_cid_next, A_diag, E, G, J_diag, Iy_diag, Iz_diag, 2); ele_tag_counter += 1
            ops.element('elasticBeamColumn', ele_tag_counter, n_cid, n_csd_next, A_diag, E, G, J_diag, Iy_diag, Iz_diag, 2); ele_tag_counter += 1

print(f"Modelo completo generado. Total Nodos: {node_tag_counter-1}, Total Elementos: {ele_tag_counter-1}")

# Visualizar el modelo completo
plt.figure(figsize=(15, 10))
opsv.plot_model(show_nodes=False, show_ele_labels=False, title="Modelo Completo de 6 Armaduras")
plt.show()

## 6. Definición de Apoyos, Cargas y Análisis

In [ ]:
# Definición de Apoyos
# Se fijarán los nodos de la base (estación 0 y estación final) de cada armadura.
print("Aplicando restricciones de apoyo (empotrado) en la base de cada armadura...")

for frame_idx in range(N):
    # Nodos de la base en la estación 0
    n_cst_start = all_truss_nodes[frame_idx][0, 'CST'][0]
    n_csd_start = all_truss_nodes[frame_idx][0, 'CSD'][0]
    n_cit_start = all_truss_nodes[frame_idx][0, 'CIT'][0]
    n_cid_start = all_truss_nodes[frame_idx][0, 'CID'][0]
    
    # Nodos de la base en la estación final (num_sections)
    n_cst_end = all_truss_nodes[frame_idx][num_sections, 'CST'][0]
    n_csd_end = all_truss_nodes[frame_idx][num_sections, 'CSD'][0]
    n_cit_end = all_truss_nodes[frame_idx][num_sections, 'CIT'][0]
    n_cid_end = all_truss_nodes[frame_idx][num_sections, 'CID'][0]
    
    base_nodes = [n_cst_start, n_csd_start, n_cit_start, n_cid_start, 
                  n_cst_end, n_csd_end, n_cit_end, n_cid_end]
    
    for node_tag in base_nodes:
        ops.fix(node_tag, 1, 1, 1, 1, 1, 1) # Empotrado

print("Apoyos definidos.")

In [ ]:
# Definición de Cargas
# Carga muerta total (pp + monten/lámina) = 1,220 N/m
# Se aplicará a los cordones superiores.
w_total_dead = -1220.0 # N/m (negativa para que actúe hacia abajo, en Z global)

# Carga de viento = 1,947 N/m
# Se aplicará lateralmente (en Y global) a los cordones delanteros (CSD y CID)
w_wind = 1947.0 # N/m (en Y positivo global)

print("Aplicando cargas muertas y de viento...")

# Crear patrones de carga
ops.timeSeries('Linear', 1)
ops.pattern('Plain', 1, 1) # Patrón para carga muerta

ops.timeSeries('Linear', 2)
ops.pattern('Plain', 2, 2) # Patrón para carga de viento

# Aplicar Carga Muerta
ops.pattern('Plain', 1, 1)
for ele in ops.getEleTags():
    # Identificar si es un cordón superior por sus nodos
    node_i, node_j = ops.eleNodes(ele)
    # Esto es ineficiente. Es mejor guardar las etiquetas de los elementos al crearlos.
    # Por ahora, una aproximación: aplicar a todos los elementos una fracción de la carga.
    # Una mejor forma: iterar sobre los elementos de cordón superior guardados.
    # Para este ejemplo, aplicaremos la carga a los cordones superiores (CSD y CST).
    pass # Se implementará de forma más robusta abajo.

# Una forma más robusta de aplicar cargas:
ele_tags_csd = []
ele_tags_cst = []
ele_tags_cid = [] # para el viento

current_ele_tag = 1
for frame_idx in range(N):
    for i in range(num_sections):
        ele_tags_cst.append(current_ele_tag); current_ele_tag += 1 # CST
        ele_tags_csd.append(current_ele_tag); current_ele_tag += 1 # CSD
        ele_tags_cid.append(current_ele_tag); current_ele_tag += 1 # CIT
        current_ele_tag += 1 # CID
    current_ele_tag += (num_sections + 1) * 4 # Saltar elementos del cajón
    current_ele_tag += (num_sections) * 4 # Saltar diagonales

# Aplicar carga muerta a los cordones superiores (CSD y CST)
w_dead_per_cordon = w_total_dead / 2.0
for ele_tag in ele_tags_cst + ele_tags_csd:
    # La carga es en Z global, pero el comando la necesita en coordenadas locales.
    # Con la transf 1 (vecxz=[0,0,1]), el eje z local es el Z global.
    ops.eleLoad('-ele', ele_tag, '-type', '-beamUniform', 0.0, w_dead_per_cordon, 0.0)

print(f"Carga muerta aplicada a {len(ele_tags_cst) + len(ele_tags_csd)} elementos de cordón superior.")

# Aplicar Carga de Viento
ops.pattern('Plain', 2, 2)
w_wind_per_cordon = w_wind / 2.0
for ele_tag in ele_tags_csd + ele_tags_cid: # Cordones delanteros
    # La carga es en Y global. El eje y local (con transf 1) es Y global.
    ops.eleLoad('-ele', ele_tag, '-type', '-beamUniform', w_wind_per_cordon, 0.0, 0.0)

print(f"Carga de viento aplicada a {len(ele_tags_csd) + len(ele_tags_cid)} elementos de cordón delantero.")

In [ ]:
# Configuración y Ejecución del Análisis Estático
print("\nConfigurando y ejecutando análisis estático...")

# Crear un caso de carga combinado (1.0*Dead + 1.0*Wind)
ops.pattern('Plain', 3, 1) # Nuevo patrón para la combinación
ops.loadConst('-time', 0.0) # Poner a cero las cargas de los patrones 1 y 2
ops.loadConst({pattern: 1, time: 1.0}) # No es un comando válido. Se analiza por separado.

# Analizaremos para el patrón de carga muerta (1) primero
ops.system('BandGeneral')
ops.numberer('RCM')
ops.constraints('Transformation')
ops.integrator('LoadControl', 1.0)
ops.algorithm('Linear')
ops.analysis('Static')

print("\nAnálisis para Carga Muerta (Patrón 1)...")
ok_dead = ops.analyze(1)

if ok_dead == 0:
    print("Análisis de carga muerta completado exitosamente.")
else:
    print("Error en el análisis de carga muerta.")

# Guardar resultados de carga muerta si es necesario y luego analizar para viento
# Por simplicidad, aquí solo se mostrarán los resultados del último análisis.

print("\nAnálisis para Carga de Viento (Patrón 2)...")
# Reiniciar el tiempo y análisis para el siguiente caso de carga
ops.setTime(0.0)
ops.integrator('LoadControl', 1.0) # Re-definir para el nuevo patrón
ok_wind = ops.analyze(1, pattern=2) # No es un comando válido, se debe activar el patrón

# La forma correcta de analizar múltiples casos de carga es un poco más compleja.
# Para este ejemplo, haremos un solo análisis con ambas cargas aplicadas en el mismo patrón.

print("\n--- REINICIANDO ANÁLISIS PARA CARGA COMBINADA ---")
ops.wipeAnalysis()
ops.setTime(0.0)

ops.pattern('Plain', 1, 1) # Usaremos solo el patrón 1 para la combinación
# Aplicar carga muerta
for ele_tag in ele_tags_cst + ele_tags_csd:
    ops.eleLoad('-ele', ele_tag, '-type', '-beamUniform', 0.0, w_dead_per_cordon, 0.0)
# Aplicar carga de viento
for ele_tag in ele_tags_csd + ele_tags_cid:
    ops.eleLoad('-ele', ele_tag, '-type', '-beamUniform', w_wind_per_cordon, 0.0, 0.0)

print("Cargas combinadas aplicadas al Patrón 1.")

# Re-correr el análisis
ops.system('BandGeneral')
ops.numberer('RCM')
ops.constraints('Transformation')
ops.integrator('LoadControl', 1.0)
ops.algorithm('Linear')
ops.analysis('Static')

ok = ops.analyze(1)

if ok == 0:
    print("Análisis estático para cargas combinadas completado exitosamente.")
else:
    print("Error en el análisis estático para cargas combinadas.")

## 7. Visualización de Resultados 3D

In [ ]:
# Visualización de Resultados Estáticos
if ok == 0:
    print("\nVisualizando resultados del análisis estático combinado...")

    # Forma Deformada
    scale_factor = 50  # Ajustar según sea necesario
    plt.figure(figsize=(15, 10))
    opsv.plot_defo(scale_factor, overlap_undisplaced=True, elementy_undisplaced='gray')
    plt.title(f"Forma Deformada (Escala: {scale_factor})")
    plt.show()

    # Diagrama de Fuerza Axial (N)
    axial_scale = 0.0001 # Ajustar escala para una visualización clara
    plt.figure(figsize=(15, 10))
    opsv.plot_section_force_diagram_3d('N', scale=axial_scale)
    plt.title(f"Diagrama de Fuerza Axial (N) (Escala: {axial_scale})")
    plt.show()

    # Diagrama de Momento Flector (Mz - momento fuerte alrededor del eje z local)
    moment_scale = 0.001 # Ajustar escala
    plt.figure(figsize=(15, 10))
    opsv.plot_section_force_diagram_3d('Mz', scale=moment_scale)
    plt.title(f"Diagrama de Momento Flector (Mz) (Escala: {moment_scale})")
    plt.show()

    # Diagrama de Momento Flector (My - momento débil alrededor del eje y local)
    plt.figure(figsize=(15, 10))
    opsv.plot_section_force_diagram_3d('My', scale=moment_scale)
    plt.title(f"Diagrama de Momento Flector (My) (Escala: {moment_scale})")
    plt.show()
    
    # Diagrama de Fuerza Cortante (Vy)
    shear_scale = 0.001 # Ajustar escala
    plt.figure(figsize=(15, 10))
    opsv.plot_section_force_diagram_3d('Vy', scale=shear_scale)
    plt.title(f"Diagrama de Fuerza Cortante (Vy) (Escala: {shear_scale})")
    plt.show()

else:
    print("No se pueden visualizar resultados debido a un error en el análisis.")